# nb_03_silver_agents

**Layer**: Silver | **Source**: `lh_bronze_insurance.agents_raw` | **Target**: `lh_silver_insurance.agents`

**Data Quality Rules**:
- Cast `hire_date` to DateType
- Require non-null: `agent_id`, `first_name`, `last_name`, `region`
- Deduplicate on `agent_id` (keep latest `_ingested_at`)
- Standardize `status` to lowercase
- Rejects → `agents_quarantine`

**Idempotency**: Uses `overwrite` mode.

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, trim, to_date, current_timestamp, lit, row_number, when, coalesce, lower
from pyspark.sql.window import Window

try:
    spark
except NameError:
    spark = SparkSession.builder.appName("nb_03_silver_agents").getOrCreate()

print("nb_03_silver_agents - Silver Layer Transformation")

In [ ]:
# ============================================================
# Step 1: Read from Bronze
# ============================================================
df_bronze = spark.table("agents_raw")
bronze_count = df_bronze.count()
print(f"Bronze rows read: {bronze_count:,}")
df_bronze.printSchema()

In [ ]:
# ============================================================
# Step 2: Cast types & clean
# ============================================================
df_typed = df_bronze \
    .withColumn("agent_id", trim(col("agent_id"))) \
    .withColumn("first_name", trim(col("first_name"))) \
    .withColumn("last_name", trim(col("last_name"))) \
    .withColumn("region", trim(col("region"))) \
    .withColumn("license_number", trim(col("license_number"))) \
    .withColumn("hire_date", to_date(col("hire_date"), "yyyy-MM-dd")) \
    .withColumn("status", lower(trim(col("status"))))

print("Type casting complete.")

In [ ]:
# ============================================================
# Step 3: Validate & route rejects
# ============================================================
REQUIRED_FIELDS = ["agent_id", "first_name", "last_name", "region"]

rejection_conditions = [
    when((col(f).isNull()) | (col(f) == ""), lit(f"null_{f}"))
    for f in REQUIRED_FIELDS
]

df_validated = df_typed.withColumn("_rejection_reason", coalesce(*rejection_conditions))

df_valid = df_validated.filter(col("_rejection_reason").isNull()).drop("_rejection_reason")
df_rejected = df_validated.filter(col("_rejection_reason").isNotNull())

valid_count = df_valid.count()
rejected_count = df_rejected.count()
print(f"Valid: {valid_count:,} | Rejected: {rejected_count:,}")

In [ ]:
# ============================================================
# Step 4: Deduplicate on agent_id
# ============================================================
window_spec = Window.partitionBy("agent_id").orderBy(col("_ingested_at").desc())

df_deduped = df_valid \
    .withColumn("_row_num", row_number().over(window_spec)) \
    .filter(col("_row_num") == 1) \
    .drop("_row_num")

deduped_count = df_deduped.count()
dupes_removed = valid_count - deduped_count
print(f"After dedup: {deduped_count:,} ({dupes_removed} duplicates removed)")

In [ ]:
# ============================================================
# Step 5: Write Silver & Quarantine
# ============================================================
silver_columns = ["agent_id", "first_name", "last_name", "region",
                  "license_number", "hire_date", "status"]

df_silver = df_deduped.select(silver_columns) \
    .withColumn("_processed_at", current_timestamp())

df_silver.write.format("delta").mode("overwrite") \
    .option("overwriteSchema", "true").saveAsTable("agents")
print(f"✓ {deduped_count:,} rows → agents")

if rejected_count > 0:
    df_rejected.withColumn("_quarantined_at", current_timestamp()) \
        .write.format("delta").mode("overwrite") \
        .option("overwriteSchema", "true").saveAsTable("agents_quarantine")
    print(f"✓ {rejected_count:,} rows → agents_quarantine")

In [ ]:
# ============================================================
# Validation
# ============================================================
print("\nSILVER AGENTS - SUMMARY")
print("=" * 50)
print(f"  Bronze input:       {bronze_count:>6,}")
print(f"  Rejected:           {rejected_count:>6,}")
print(f"  Duplicates removed: {dupes_removed:>6,}")
print(f"  Silver output:      {deduped_count:>6,}")
print(f"  Quality rate:       {(deduped_count/bronze_count*100):.1f}%")

assert spark.table("agents").count() == deduped_count
print("\n✓ Verification passed")
spark.table("agents").show(5, truncate=25)